# Class 3 — Kafka and Amazon MSK (EMR)

This notebook explains **what Kafka is, why it exists, how Amazon MSK fits in, and how Spark Structured Streaming reads/writes it**. It is conceptual first; a runnable local demo follows using a filesystem stand-in (real MSK connectivity requires network/broker setup covered in `infra/terraform/` and `emr-notebooks/02_kafka_msk_streaming_ingest.ipynb`).

## 1. What problem Kafka solves

Before event streaming platforms, systems integrated point-to-point: service A calls service B's API, or writes directly to B's database. This doesn't scale past a handful of systems (N systems need up to N*(N-1) integrations) and couples producers to consumers' availability.

Kafka decouples producers from consumers with a **durable, ordered, replayable log**:

```text
 Producers                     Kafka topic (durable log)                 Consumers
 ---------                     --------------------------                ---------
 web app        -----\                                          /-----> Spark Structured Streaming
 mobile app     ------>  [ partition 0: msg0 msg1 msg2 msg3 ... ] ------> fraud detection service
 IoT device     -----/    [ partition 1: msg0 msg1 msg2 ...     ] \-----> analytics dashboard
                          [ partition 2: msg0 msg1 ...          ]
```

Producers write once; any number of independent consumers can read the same data, each at their own pace, each remembering their own position (offset).

## 2. Core concepts

- **Topic** -- a named stream of events (e.g. `retail-clickstream`).
- **Partition** -- a topic is split into ordered, append-only partitions. Order is guaranteed *within* a partition, not across partitions of the same topic.
- **Offset** -- the position of a message within a partition.
- **Broker** -- a Kafka server that stores partitions and serves reads/writes.
- **Replication factor** -- each partition is copied to N brokers for durability.
- **Consumer group** -- a set of consumers that split a topic's partitions between them.
- **Key** -- an optional per-message key. Messages with the same key always land in the same partition, which is how you get ordering guarantees *per entity*.

## 3. Delivery/ordering guarantees that matter for Spark integration

- Kafka guarantees **at-least-once** delivery to consumers by default.
- Spark Structured Streaming turns this into **effectively-exactly-once** end to end by tracking consumed offsets in its own checkpoint and writing to an idempotent/transactional sink (Delta Lake).
- Ordering is only guaranteed **within a partition**.

## 4. Amazon MSK (Managed Streaming for Apache Kafka)

MSK is AWS's managed Kafka service. Two flavors: **MSK Provisioned** (choose broker instance type/count/storage, billed per broker-hour) and **MSK Serverless** (capacity scales automatically, billed per partition-hour and GB in/out/retained -- simpler to provision, which is why `infra/terraform/` defaults to it).

What you provide either way: **networking** (VPC/subnet reachability, security groups on the broker port), **authentication** (IAM auth, SASL/SCRAM, mutual TLS, or plaintext dev-only), and **topics** (created via the Kafka Admin API/CLI or Terraform).

## 5. Spark <-> Kafka: the actual read/write shape

```python
raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", "retail-clickstream")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)
```

The resulting DataFrame always has this fixed schema:

| column | type | meaning |
|---|---|---|
| `key` | binary | producer-supplied partition key |
| `value` | binary | the actual message payload (commonly JSON or Avro) |
| `topic` | string | topic name |
| `partition` | int | partition number |
| `offset` | long | offset within the partition |
| `timestamp` | timestamp | broker-assigned or producer-assigned event timestamp |
| `timestampType` | int | 0 = create time, 1 = log append time |

You cast `value` to string and parse it with `from_json(col, schema)` against a known schema, exactly like the demo below.

## 6. Runnable demo without live MSK

A JSON-lines directory is a reasonable stand-in for teaching the *parsing and schema* half of Kafka integration without requiring a live broker. `emr-notebooks/02_kafka_msk_streaming_ingest.ipynb` uses the real `kafka` source against MSK; this cell only demonstrates schema parsing and bronze-table semantics.

Run the cell below first -- it configures Delta Lake for this notebook's Spark session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "class_emr"
base_path = "s3://<your-lakehouse-bucket>/data/class-emr"

def table(name):
    return f"`{schema}`.`{name}`"

def path(*parts):
    return base_path.rstrip("/") + "/" + "/".join(p.strip("/") for p in parts)

def checkpoint(name):
    return path("checkpoints", name)

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType
import json
import random
from datetime import datetime, timedelta, timezone

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema}` LOCATION '{path('tables')}'")
spark.sql(f"USE `{schema}`")

## Step 1 — Simulate Kafka's on-wire shape

We write JSON files that mimic Kafka `value` payloads (a real MSK payload would be the same JSON bytes, just delivered via the `kafka` source instead of files). Writing straight to the S3 path works directly from a Jupyter/Sparkmagic session -- no local-filesystem staging step needed.

In [ ]:
click_event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_ts", TimestampType(), False),
    StructField("user_id", StringType(), False),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
])

random.seed(3)
start = datetime.now(timezone.utc) - timedelta(minutes=30)
rows = []
for i in range(500):
    event_type = random.choice(["view", "add_to_cart", "purchase", "search", "checkout"])
    qty = random.randint(1, 3) if event_type in ("purchase", "checkout") else None
    price = round(random.uniform(5, 200), 2) if qty else None
    rows.append({
        "event_id": f"kevt-{i:06d}",
        "event_ts": (start + timedelta(seconds=random.randint(0, 1800))).isoformat(),
        "user_id": f"u{random.randint(1, 200):05d}",
        "product_id": f"p{random.randint(1, 50):04d}",
        "event_type": event_type,
        "quantity": qty,
        "price": price,
    })

sim_topic_path = path("kafka_sim", "retail-clickstream")
spark.createDataFrame([(json.dumps(r),) for r in rows], "json_payload string").coalesce(1).write.mode("overwrite").text(sim_topic_path)
print("Wrote simulated topic files to", sim_topic_path)

## Step 2 — Parse the payload against a known schema (the part that's identical for real Kafka)

`spark.readStream.format("text")` here stands in for `spark.readStream.format("kafka")`. Everything from `from_json` onward is **exactly** what the real MSK notebook does to the `value` column.

In [ ]:
raw_stream = (
    spark.readStream.format("text").load(sim_topic_path)
    .withColumnRenamed("value", "json_payload")
)

parsed = (
    raw_stream
    .withColumn("event", F.from_json(F.col("json_payload"), click_event_schema))
    .select("json_payload", "event.*")
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source", F.lit("kafka_msk_simulated"))
)

query = (
    parsed.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint("class_kafka_sim_bronze"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table("class_kafka_sim_bronze"))
)
query.awaitTermination()

spark.table(table("class_kafka_sim_bronze")).orderBy(F.desc("event_ts")).limit(10).toPandas()

## What differs when this points at real MSK

Only the source block changes -- the parsing/business logic is identical:

```python
raw_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", "retail-clickstream")
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "AWS_MSK_IAM")
    .option("kafka.sasl.jaas.config",
            "software.amazon.msk.auth.iam.IAMLoginModule required;")
    .option("kafka.sasl.client.callback.handler.class",
            "software.amazon.msk.auth.iam.IAMClientCallbackHandler")
    .load()
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING) AS json_payload", "timestamp AS kafka_ts")
)
```

This requires the `aws-msk-iam-auth` and `spark-sql-kafka-0-10` packages on the classpath (add them to the `%%configure` cell's `spark.jars.packages`, the same mechanism `emr-notebooks/02_kafka_msk_streaming_ingest.ipynb` uses), and network reachability from the EMR cluster's VPC/subnets to the MSK brokers' security groups -- see `infra/terraform/msk.tf` and `RUNBOOK.md`.

Unlike the Databricks version of this stack, EMR needs no service-credential indirection for MSK IAM auth -- the EC2 instance profile Plan 1's `iam.tf` attaches supplies AWS credentials ambiently, exactly like the classic (pre-serverless) approach this markdown describes.

## What's next

`04_data_lakehouse_delta_s3.ipynb` covers what happens on the *write* side once events are flowing: why raw files on S3 aren't enough, and what Delta Lake adds.